# 1 · Digitising the transcriptions (OCR)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chemvatho/kolsch-tandem/blob/main/01_ocr/01_ocr_digitisation.ipynb)

**Pipeline stage 1 of 6 — Kölsch ASR.** Convert the printed *Alles Kölsch*
(Bhatt & Lindlar, 1998) transcription pages into clean, machine-readable
Unicode text, preserving the dialectal orthography.

We evaluate three OCR engines in increasing order of robustness and keep the
best output:

1. **Tesseract** — open-source baseline (offline, free).
2. **EasyOCR** — deep-learning OCR, better on mixed fonts.
3. **Gemini 2.5 Pro** — vision-language model that reads pages *in context*;
   selected for production because it preserves elision apostrophes
   (`d'r`, `m'r`, `ha'mer`), dialectal spellings and special characters.

> Input: a folder of scanned page images. Output: a mirrored folder of `.txt`
> files with the faithful transcription.

## Setup

In [ ]:
# Colab / local install
!pip -q install pytesseract easyocr google-generativeai pillow
# Tesseract binary + German language pack (Colab/Ubuntu)
!apt-get -qq install -y tesseract-ocr tesseract-ocr-deu >/dev/null
import os, glob, pathlib
from PIL import Image
print("ready")

In [ ]:
# Point this at your scanned pages (one image per page).
PAGES_DIR  = "pages"          # input: .png/.jpg scans
OUT_DIR    = "ocr_txt"        # output: .txt per page
os.makedirs(OUT_DIR, exist_ok=True)
# For a quick demo with no data, drop a couple of images into ./pages first.
sample = sorted(glob.glob(f"{PAGES_DIR}/*.png") + glob.glob(f"{PAGES_DIR}/*.jpg"))
print(len(sample), "page images found")

## 1 · Tesseract (baseline)

Offline and free, but inconsistent on the elision apostrophes and dialectal
diacritics that are linguistically essential in Kölsch.

In [ ]:
import pytesseract

def ocr_tesseract(img_path, lang="deu"):
    return pytesseract.image_to_string(Image.open(img_path), lang=lang)

if sample:
    print(ocr_tesseract(sample[0])[:500])

## 2 · EasyOCR

Copes better with mixed fonts and page layout, but still struggles with
dialect-specific spellings and special characters.

In [ ]:
import easyocr
reader = easyocr.Reader(["de"], gpu=True)   # set gpu=False on CPU-only machines

def ocr_easyocr(img_path):
    # detail=0 returns plain strings; paragraph=True groups lines
    lines = reader.readtext(img_path, detail=0, paragraph=True)
    return "\n".join(lines)

if sample:
    print(ocr_easyocr(sample[0])[:500])

## 3 · Gemini 2.5 Pro  (selected)

A vision-language model reads each page *with contextual understanding* rather
than glyph-by-glyph, so it keeps apostrophes, dialectal spellings and special
characters. The prompt instructs the model to transcribe **faithfully** and to
**not** correct or normalise the dialect — the standard failure mode of
LLM-based OCR on non-standard text.

Get a key at <https://aistudio.google.com/app/apikey> and set it below.

In [ ]:
import google.generativeai as genai
from google.colab import userdata  # on Colab; else use os.environ

# GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "PASTE_YOUR_KEY")
genai.configure(api_key=GEMINI_API_KEY)

OCR_PROMPT = (
    "You are an OCR system for German dialect (Kölsch) text. "
    "Read all text from this image and output clean Unicode text. "
    "Do NOT translate or correct the dialect spelling. "
    "Preserve apostrophes, special characters, paragraph and line breaks. "
    "Do NOT explain anything. Output the text as plain text only."
)

def ocr_gemini(img_path, model_name="gemini-2.5-pro"):
    model = genai.GenerativeModel(model_name)
    img = Image.open(img_path)
    resp = model.generate_content([OCR_PROMPT, img])
    return resp.text

if sample:
    print(ocr_gemini(sample[0])[:500])

## 4 · Batch driver — process the whole page tree with Gemini

In [ ]:
import time

def batch_ocr(pages_dir=PAGES_DIR, out_dir=OUT_DIR, engine=ocr_gemini, overwrite=False):
    imgs = sorted(glob.glob(f"{pages_dir}/**/*.png", recursive=True) +
                  glob.glob(f"{pages_dir}/**/*.jpg", recursive=True))
    for i, img in enumerate(imgs, 1):
        rel = os.path.relpath(img, pages_dir)
        out = os.path.join(out_dir, os.path.splitext(rel)[0] + ".txt")
        os.makedirs(os.path.dirname(out), exist_ok=True)
        if os.path.exists(out) and not overwrite:
            continue
        try:
            text = engine(img)
            with open(out, "w", encoding="utf-8") as f:
                f.write(text)
            print(f"[{i}/{len(imgs)}] {rel}  ({len(text)} chars)")
        except Exception as e:
            print(f"[{i}/{len(imgs)}] {rel}  ERROR: {e}")
        time.sleep(0.5)   # be gentle with the API
    print("done ->", out_dir)

# batch_ocr()   # uncomment to run over ./pages

## 5 · Final human pass

OCR is never perfect on dialect text. After the batch run, a human proofreads
each `.txt` against the source page — correcting residual errors and confirming
that elisions and special characters survived. The corrected `.txt` tree is the
input to **Notebook 2 (corpus statistics)** and **Notebook 3 (segmentation)**.

### Why Gemini was selected
| Engine | Strength | Weakness on Kölsch |
|---|---|---|
| Tesseract | offline, free | drops `'` elisions, mishandles diacritics |
| EasyOCR | good on layout | weak on dialect spellings / special chars |
| **Gemini 2.5 Pro** | contextual, keeps orthography | needs an API key (small cost) |
